# Module 6 — Matplotlib and Seaborn

Run in order. Data is synthetic and intended for learning.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
path = Path('data/retail_sales_synthetic.csv')
if not path.exists(): path = Path('retail_sales_synthetic.csv')  # Colab upload fallback
df = pd.read_csv(path, parse_dates=['order_date'])
assert df.order_id.is_unique
print(df.shape)
display(df.head())

## 1. Data audit and KPIs

In [ ]:
print(df.isna().sum())
print(df[['revenue','profit','units']].describe())
revenue = df.revenue.sum(); profit = df.profit.sum()
print({'orders':len(df), 'revenue':round(revenue,2),'profit':round(profit,2),'margin_pct':round(100*profit/revenue,2)})

## 2. Revenue by category — sorted bar chart

In [ ]:
by_cat = df.groupby('category',as_index=False)['revenue'].sum().sort_values('revenue')
fig,ax=plt.subplots(figsize=(8,4))
ax.barh(by_cat.category,by_cat.revenue)
ax.set(title='Revenue by category (synthetic orders)',xlabel='Revenue (USD)')
fig.tight_layout();plt.show()

## 3. Monthly revenue and profit — time series

In [ ]:
monthly = df.set_index('order_date').resample('MS')[['revenue','profit']].sum()
fig,ax=plt.subplots(figsize=(9,4))
monthly.plot(ax=ax,marker='o')
ax.set(title='Monthly revenue and profit',xlabel='Month',ylabel='USD')
fig.tight_layout();plt.show()

## 4. Order-value distribution — histogram

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
sns.histplot(data=df,x='revenue',bins=25,ax=ax)
ax.set(title='Order revenue distribution',xlabel='Order revenue (USD)')
fig.tight_layout();plt.show()

## 5. Revenue distribution by region — boxplot

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
sns.boxplot(data=df,x='region',y='revenue',ax=ax)
ax.set(title='Order revenue by region',ylabel='Revenue per order (USD)')
fig.tight_layout();plt.show()

## 6. Relationship between revenue and profit — scatter

In [ ]:
fig,ax=plt.subplots(figsize=(8,4))
sns.scatterplot(data=df,x='revenue',y='profit',hue='channel',alpha=.6,ax=ax)
ax.set(title='Order revenue vs profit (association, not causation)')
fig.tight_layout();plt.show()

## 7. Correlation heatmap

In [ ]:
corr=df[['units','unit_price','discount_rate','revenue','cost','profit']].corr()
fig,ax=plt.subplots(figsize=(7,5))
sns.heatmap(corr,annot=True,fmt='.2f',cmap='coolwarm',center=0,ax=ax)
ax.set(title='Numeric feature correlations')
fig.tight_layout();plt.show()

## 8. Category by region — heatmap

In [ ]:
matrix=df.pivot_table(index='category',columns='region',values='revenue',aggfunc='sum',fill_value=0)
fig,ax=plt.subplots(figsize=(8,4))
sns.heatmap(matrix,annot=True,fmt='.0f',cmap='Blues',ax=ax)
ax.set(title='Revenue by category and region (USD)')
fig.tight_layout();plt.show()

## 9. Channel comparison and profit margin

In [ ]:
by_channel=df.groupby('channel').agg(revenue=('revenue','sum'),profit=('profit','sum'),orders=('order_id','count'))
by_channel['margin_pct']=100*by_channel.profit/by_channel.revenue
print(by_channel)
fig,ax=plt.subplots(figsize=(7,4))
by_channel['margin_pct'].plot.bar(ax=ax)
ax.set(title='Weighted profit margin by sales channel',ylabel='Margin (%)',xlabel='Channel')
fig.tight_layout();plt.show()

## 10. Save a chart and a summary table

In [ ]:
out=Path('outputs');out.mkdir(exist_ok=True)
fig,ax=plt.subplots(figsize=(8,4))
ax.barh(by_cat.category,by_cat.revenue)
ax.set(title='Category revenue',xlabel='USD')
fig.tight_layout();fig.savefig(out/'category_revenue.png',dpi=160)
plt.close(fig)
by_cat.to_csv(out/'category_revenue.csv',index=False)
print('Saved:',out/'category_revenue.png',out/'category_revenue.csv')